## Notebook Description

### Dataset Description
The data includes EV charging reports, Hourly EV charging loads, and idle capacity, for all sessions and users individually along with weather and traffic data of the location.
The datasets are described in the Data In Brief article ["Residential electric vehicle charging datasets from apartment buildings" (2021)](https://www.sciencedirect.com/science/article/pii/S2352340921003899)

### Notebook purpose
- Practice usage of unsupervised learning algorithms:
    - Session-level clustering
    - User-level clustering
    - Anomaly detection

### Workflow
- Environment creation
- Data collection
- Data preparation
- Exploratory Data Analysis
- Feature Engineering
- Model development
- Results

Step 0: Environment Creation

In [27]:
#%pip install numpy; pandas; matplotlib; seaborn; scikit-learn; kagglehub

In [28]:
import numpy as np
import pandas as pd
import sklearn
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns


### Step 1: Data collection

In [29]:
from pathlib import Path

output_dir = Path('datasets')
output_dir.mkdir(exist_ok=True)

path = kagglehub.dataset_download("anshtanwar/residential-ev-chargingfrom-apartment-buildings", output_dir=output_dir)

print("Path to dataset files:", path)

csv_path = output_dir / 'archive'
print("Path to the csv files", csv_path)

Path to dataset files: datasets
Path to the csv files datasets\archive


In [30]:

df1 = pd.read_csv(csv_path / 'Dataset 1_EV charging reports.csv', sep=';')
df1.head()



,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,Duration_hours,month_plugin,weekdays_plugin,Plugin_category,Duration_category
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3","0,05",Dec,Friday,late morning (9-12),Less than 3 hours
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,10,21.12.2018 10:32,10.0,"0,87","0,136666667",Dec,Friday,late morning (9-12),Less than 3 hours
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,11,21.12.2018 19:46,19.0,"29,87","8,216388889",Dec,Friday,late morning (9-12),Between 6 and 9 hours
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,16,23.12.2018 16:40,16.0,"15,56","24,41972222",Dec,Saturday,late afternoon (15-18),More than 18 hours
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,22,24.12.2018 23:02,23.0,"3,62","0,970555556",Dec,Monday,late evening (21-midnight),Less than 3 hours


#### Dataset 1: EV Charging Reports
The CSV file “Dataset 1” describes 6,878 individual charging sessions, registered by 97 user IDs from December 2018 to January 2020.

<details>
<summary>Dataset 1: EV charging reports, describing each individual EV charging session.</summary>

| Column | Description |
|---|---|
| `session_ID` | Unique ID for EV charging session (N=6878) |
| `Garage_ID` | ID for garage address (N=24) |
| `User_ID` | ID for user (N=97) |
| `User_type` | CP ownership: private or shared CPs |
| `Shared_ID` | ID for shared CP when used (N=12) |
| `Start_plugin` | Plug-in date and time (format: 21.12.2018 10:20) |
| `Start_plugin_hour` | Clock hour for plug-in (00–23) |
| `End_plugout` | Plug-out date and time (format: 21.12.2018 10:20) |
| `End_charging_hour` | Clock hour for plug-out (00–23) |
| `El_kWh` | Charged energy (kWh) |

</details>

In [31]:
df2 = pd.read_csv(csv_path / 'Dataset 1_EV charging reports.csv', sep=';')
df2.head()

,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,Duration_hours,month_plugin,weekdays_plugin,Plugin_category,Duration_category
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3","0,05",Dec,Friday,late morning (9-12),Less than 3 hours
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,10,21.12.2018 10:32,10.0,"0,87","0,136666667",Dec,Friday,late morning (9-12),Less than 3 hours
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,11,21.12.2018 19:46,19.0,"29,87","8,216388889",Dec,Friday,late morning (9-12),Between 6 and 9 hours
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,16,23.12.2018 16:40,16.0,"15,56","24,41972222",Dec,Saturday,late afternoon (15-18),More than 18 hours
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,22,24.12.2018 23:02,23.0,"3,62","0,970555556",Dec,Monday,late evening (21-midnight),Less than 3 hours


<details>
<summary>Dataset 2: Hourly EV charging loads and idle capacity, for all users individually.</summary>

| Column | Description |
|---|---|
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `date_to` | Ending time (format: 22.01.2019 20:00) |
| `User_ID` | ID for user (N=97) |
| `session_ID` | Unique ID for EV charging session (N=6878) |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power (ref. [1]), for users individually |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power (ref. [1]), for users individually |
| `Flex_3_6kW` | Synthetic hourly idle capacity (kWh/h) assuming 3.6 kW charging power, for users individually |
| `Flex_7_2kW` | Synthetic hourly idle capacity (kWh/h) assuming 7.2 kW charging power, for users individually |

</details>

In [32]:
df3a = pd.read_csv(csv_path / 'Dataset 3a_Hourly EV loads - Aggregated private.csv', sep=';')
df3a.head()

,date_from,daily_hour,weekday,month,Synthetic_3_6kW,Synthetic_7_2kW,Flex_3_6kW,Flex_7_2kW,n_private
0,21.12.2018 10:00,10,Friday,Dec,"1,17","1,17",0,"0,174",1
1,21.12.2018 11:00,11,Friday,Dec,"1,62","3,24",0,0,1
2,21.12.2018 12:00,12,Friday,Dec,"3,6","7,2",0,0,1
3,21.12.2018 13:00,13,Friday,Dec,"3,6","7,2",0,0,1
4,21.12.2018 14:00,14,Friday,Dec,"3,6","7,2",0,0,1


In [33]:
df3b = pd.read_csv(csv_path / 'Dataset 3b_Hourly EV loads - Aggregated shared.csv', sep=';')
df3b.head()

,date_from,daily_hour,weekday,month,Synthetic_3_6kW,Synthetic_7_2kW,Flex_3_6kW,Flex_7_2kW,n_shared
0,10.01.2019 00:00,0,Thursday,Jan,0,0,0,0,1
1,10.01.2019 01:00,1,Thursday,Jan,0,0,0,0,1
2,10.01.2019 02:00,2,Thursday,Jan,0,0,0,0,1
3,10.01.2019 03:00,3,Thursday,Jan,0,0,0,0,1
4,10.01.2019 04:00,4,Thursday,Jan,0,0,0,0,1


<details>
<summary>Dataset 3: Hourly EV charging loads and idle capacity, aggregated for private or shared CPs</summary>

| Column | Description |
|---|---|
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `daily_hour` | Clock hour (00–23) |
| `weekday` | Weekday (Monday–Sunday) |
| `month` | Month (January–December) |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Flex_3_6kW` | Synthetic hourly idle capacity (kWh/h) assuming 3.6 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Flex_7_2kW` | Synthetic hourly idle capacity (kWh/h) assuming 7.2 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `2a: n_private` | Number of registered User IDs using private CPs (increasing, 1 to 58) |
| `2b: n_shared` | Number of registered User IDs using shared CPs (increasing, 1 to 24) |

</details>

In [34]:
df5 = pd.read_csv(csv_path / 'Dataset 5_AMS data from garage Bl2.csv', sep=';')
df5.head()

,date_from,date_to,month,AMS_kWh,Synthetic_3_6kW,Synthetic_7_2kW,Simultaneous_if_3_6kW
0,02.01.2019 19:00,02.01.2019 20:00,Jan,"0,05","0,06","0,12",1.0
1,02.01.2019 20:00,02.01.2019 21:00,Jan,"6,6","3,6","7,2",1.0
2,02.01.2019 21:00,02.01.2019 22:00,Jan,"6,8","3,6","7,2",1.0
3,02.01.2019 22:00,02.01.2019 23:00,Jan,"1,45","3,6","0,12",1.0
4,02.01.2019 23:00,03.01.2019 00:00,Jan,"0,05","3,6",0,1.0


Dataset 5: Hourly smart meter data from garage Bl2.
<details>
<summary>Column descriptions</summary>

| Column | Description |
|---|---|
| `Garage_ID` | ID for garage address (Bl2) |
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `date_to` | Ending time (format: 22.01.2019 20:00) |
| `month` | Measurement starting month |
| `AMS_kWh` | Aggregated electricity use in the garage each hour, measured by AMS meter |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power, aggregated for users in the garage |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power, aggregated for users in the garage |
| `Simultaneous_if_3_6kW` | Number of simultaneous charging sessions, assuming all sessions charge with 3.6 kW charging power. NA if no charging sessions are assumed |

</details>

In [35]:
df6 = pd.read_csv(csv_path / 'Dataset 6_Local traffic distribution.csv', sep=';')
df6.head()

,Date_from,Date_to,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,01.12.2018 00:00,01.12.2018 01:00,639,0,0,4,144
1,01.12.2018 01:00,01.12.2018 02:00,487,153,115,21,83
2,01.12.2018 02:00,01.12.2018 03:00,408,85,75,10,69
3,01.12.2018 03:00,01.12.2018 04:00,282,89,56,8,39
4,01.12.2018 04:00,01.12.2018 05:00,165,64,34,3,25


<details>
<summary>Dataset 6: Local traffic density </summary>

| Column | Description |
|---|---|
| `Date_from` | Starting time (format: 22.01.2019 19:00) |
| `Date_to` | Ending time (format: 22.01.2019 20:00) |
| `Location 1 to 5` | Number of vehicles shorter than 5.6 meters each hour at five nearby traffic locations |

</details>

In [36]:
df_weather = pd.read_csv(csv_path / 'Norway_Trondheim_ExactLoc_Weather.csv')
df_weather.head()

,Unnamed: 0,clouds,datetime,dewpt,dhi,dni,ghi,max_dhi,max_dni,max_ghi,...,solar_rad,t_dhi,t_dni,t_ghi,t_solar_rad,temp,ts,wind_dir,wind_gust_spd,wind_spd
0,0,66,2018-12-01,-1.7,5,40,6,34,288,48,...,2,127,966,144,52,7.4,1543618800,187,12.8,5.5
1,1,64,2018-12-02,-2.8,5,38,6,33,281,46,...,5,123,923,135,126,5.4,1543705200,177,10.8,3.6
2,2,100,2018-12-03,-0.7,5,37,5,33,273,44,...,1,119,881,128,21,2.4,1543791600,142,4.8,1.5
3,3,93,2018-12-04,-0.5,5,35,5,32,266,42,...,1,115,842,121,22,0.8,1543878000,234,12.8,3.9
4,4,87,2018-12-05,-1.9,5,34,5,32,260,41,...,2,111,804,114,39,-0.4,1543964400,206,13.6,3.7


<details>
<summary>Dataset Weather: Weather for Trondheim</summary>

| Column | Description |
|---|---|
| `Unnamed: 0` | Row index included in the CSV; likely not a weather measurement |
| `clouds` | Cloud cover |
| `datetime` | Date of the weather record |
| `dewpt` | Dew-point temperature |
| `dhi` | Diffuse horizontal solar irradiance |
| `dni` | Direct normal solar irradiance |
| `ghi` | Global horizontal solar irradiance |
| `max_dhi` | Maximum diffuse horizontal solar irradiance |
| `max_dni` | Maximum direct normal solar irradiance |
| `max_ghi` | Maximum global horizontal solar irradiance |
| `max_temp` | Maximum temperature |
| `max_temp_ts` | Timestamp for the maximum temperature |
| `max_uv` | Maximum UV index |
| `max_wind_dir` | Wind direction when wind speed was at its maximum |
| `max_wind_spd` | Maximum wind speed |
| `max_wind_spd_ts` | Timestamp for the maximum wind speed |
| `min_temp` | Minimum temperature |
| `min_temp_ts` | Timestamp for the minimum temperature |
| `precip` | Precipitation |
| `precip_gpm` | Precipitation measurement reported as `precip_gpm` in the source data |
| `pres` | Atmospheric pressure |
| `revision_status` | Weather-data revision status |
| `rh` | Relative humidity |
| `slp` | Sea-level pressure |
| `snow` | Snowfall |
| `snow_depth` | Snow depth |
| `solar_rad` | Solar radiation |
| `t_dhi` | Total diffuse horizontal solar irradiance |
| `t_dni` | Total direct normal solar irradiance |
| `t_ghi` | Total global horizontal solar irradiance |
| `t_solar_rad` | Total solar radiation |
| `temp` | Temperature |
| `ts` | Timestamp for the weather record |
| `wind_dir` | Wind direction |
| `wind_gust_spd` | Wind gust speed |
| `wind_spd` | Wind speed |
</details>